<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/Nexus_Raven_Quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NexusRaven Quickstart
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference : https://github.com/nexusflowai/NexusRaven
## Model : https://huggingface.co/Nexusflow/NexusRaven-13B

In [ ]:
!nvidia-smi

# 라이브러리 설치

In [ ]:
# [Cell 1] Environment Setup
import torch

# 1. GPU Check
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found. Check Runtime type.")

print("\nInstalling Libraries...")

# 2. Install Libraries
# protobuf==3.20.3 : 'MessageFactory' 오류 방지를 위해 구버전으로 고정 (가장 중요!)
# transformers, accelerate, bitsandbytes : 모델 로드 및 양자화 필수 라이브러리
!pip install -U "transformers>=4.40.0" "accelerate" "bitsandbytes>=0.45.0" "protobuf==3.20.3"

print("\nSetup Completed.")

## Nexus-Raven-13B(V1) 모델 불러오기

In [ ]:
# [Cell 2] Load Nexus Raven (V1) & Set Eval Mode
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# --- Configuration ---
model_id = "Nexusflow/NexusRaven-13B"

# 1. Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f" Loading Model & Tokenizer: {model_id}...")

try:
    # 2. Load Tokenizer (아까 포함했던 부분)
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # 3. Load Model
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=False
    )

    # 4. Set to Evaluation Mode (추가된 부분!)
    # 모델을 추론 모드로 전환하여 결과를 안정적으로 만듭니다.
    model.eval()

    print(f"Model Loaded Successfully on {model.device}")
    print("Ready for Inference (Eval Mode ON)")

except Exception as e:
    print(f"Error: {e}")

In [ ]:
# do_sample : False -> 항상 가장 높은 확률값을 가진 단어로 예측
def gen(x):
    q = x
    gened = model.generate(
        **tokenizer(
            q,
            return_tensors='pt',
            return_token_type_ids=False
        ).to('cuda'),
        max_new_tokens=128,
        early_stopping=True,
        do_sample=False,
    )
    return tokenizer.decode(gened[0]).replace(q, "")

In [ ]:
prompt_template = """
<human>:
OPTION:
<func_start>def hello_world(n : int)<func_end>
<docstring_start>
\"\"\"
Prints hello world to the user.

Args:
n (int) : Number of times to print hello world.
\"\"\"
<docstring_end>

OPTION:
<func_start>def hello_universe(n : int)<func_end>
<docstring_start>
\"\"\"
Prints hello universe to the user.

Args:
n (int) : Number of times to print hello universe.
\"\"\"
<docstring_end>

User Query: Question: {question}

Please pick a function from the above options that best answers the user query and fill in the appropriate arguments.<human_end>
"""
input_prompt = prompt_template.format(question="Please print hello world 10 times.")

In [ ]:
input_prompt

##Nexus-Raven-13B Model Generation Result

In [ ]:
result = gen(input_prompt)
result

In [ ]:
# <s>
# <human>:
# OPTION:
# <func_start> def hello_world(n : int)<func_end>
# <docstring_start>
# """
# Prints hello world to the user.

# Args:
# n (int) : Number of times to print hello world.
# """
# <docstring_end>

# OPTION:
# <func_start> def hello_universe(n : int)<func_end>
# <docstring_start>
# """
# Prints hello universe to the user.

# Args:
# n (int) : Number of times to print hello universe.
# """
# <docstring_end>

# User Query: Question: Please print hello world 10 times.

# Please pick a function from the above options that best answers the user query and fill in the appropriate arguments.<human_end>
#  Thought: The purpose of the def hello_world(n : int) is to print hello world to the user.
# Initial Answer: hello_world(10)
# Reflection: The hello_world function takes in one argument, n, which is an integer. The user has asked to print hello world 10 times.

# The call provided is hello_world(10).

# The call does not need to be improved.

# This call can be run:
# Fixed Call: hello_world(10)
# Fixed Function Name: hello_world
# Fixed Function Args: {"

In [ ]:
# Get the "Initial Call" only
start_str = "Initial Answer: "
end_str = "\nReflection: "
start_idx = result.find(start_str) + len(start_str)
end_idx = result.find(end_str)
function_call = result[start_idx: end_idx]

print (f"Generated Call: {function_call}")